# NB-P01-02 — NumPy, pandas and Plotting

Companion notebook for [LP01-06](../lessons/LP01-06-numpy-pandas-and-plotting.md). Everything is
generated locally with a fixed seed, so your numbers match the lesson.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=7)
print("ready")

## 1. Arrays versus lists

An array is one block of one type with a shape; `*` means elementwise multiplication, not repetition.

In [ ]:
a = np.array([1, 2, 3])
print(a * 2)
print([1, 2, 3] * 2)
print("shape:", a.shape, "dtype:", a.dtype)

## 2. Shape and axis

`axis=0` collapses the rows (one value per column); `axis=1` collapses the columns.

In [ ]:
m = np.array([[1, 3], [2, 4]])
print("matrix:\n", m)
print("sum axis=0:", m.sum(axis=0))
print("sum axis=1:", m.sum(axis=1))
print("mean axis=0:", m.mean(axis=0).round(2))

### Broadcasting

Subtracting a column-mean vector from a matrix works even though the shapes differ: NumPy stretches the
smaller shape across the larger one. This is the mechanism behind standardisation.

In [ ]:
X = rng.normal(loc=50, scale=10, size=(6, 3)).round(1)
print(X)
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print("column means after:", Z.mean(axis=0).round(10))
print("column sds after:  ", Z.std(axis=0).round(10))

## 3. Boolean masks and reductions

In [ ]:
scores = np.array([78, 91, 64, 85, 72, 95])
mask = scores >= 80
print(mask)
print("above 80:", scores[mask])
print("count    :", mask.sum())
print("share    :", round(float(mask.mean()), 3))

## 4. pandas: a table with named columns

In [ ]:
df = pd.DataFrame({
    "student": [f"s{i:02d}" for i in range(1, 9)],
    "city": rng.choice(["Lahore", "Karachi", "Islamabad"], size=8),
    "hours": rng.uniform(2, 12, size=8).round(1),
    "score": rng.integers(45, 96, size=8),
})
df

In [ ]:
print(df.dtypes.to_string())
print()
print(df.describe().round(1).to_string())

### Select, filter, group, sort

These four operations cover most analysis. Each returns a new object, so they chain left to right.

In [ ]:
print(df[["student", "score"]].head(3).to_string(index=False))
print()
print(df[df["score"] >= 80].to_string(index=False))

In [ ]:
summary = (df.groupby("city")
             .agg(n=("score", "size"),
                  mean_score=("score", "mean"),
                  mean_hours=("hours", "mean"))
             .round(1)
             .sort_values("mean_score", ascending=False))
print(summary.to_string())

## 5. Plotting

Label every axis, and give each figure a title that says what is shown. A plot without labels is not a
figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].hist(df["score"], bins=6, edgecolor="white", color="#4c72b0")
axes[0].set_title("Score distribution")
axes[0].set_xlabel("Score")
axes[0].set_ylabel("Students")

axes[1].scatter(df["hours"], df["score"], s=40, alpha=0.8, color="#55a868")
axes[1].set_title("Score against study hours")
axes[1].set_xlabel("Hours per week")
axes[1].set_ylabel("Score")

cities = sorted(df["city"].unique())
axes[2].boxplot([df.loc[df["city"] == c, "score"] for c in cities])
axes[2].set_xticks(range(1, len(cities) + 1), cities)
axes[2].set_title("Score by city")
axes[2].set_ylabel("Score")
axes[2].tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

### The same data with seaborn

Seaborn adds statistical defaults and grouping by a column, at the cost of a little control. Use it for
exploration, Matplotlib for the final figure.

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
ax = sns.scatterplot(data=df, x="hours", y="score", hue="city", s=70)
ax.set_title("Study hours and score, coloured by city")
ax.set_xlabel("Hours per week")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## 6. Reproducibility

A fixed seed makes output identical between runs. Notice that reusing the same generator continues the
sequence rather than restarting it.

In [ ]:
def sample_means(seed, n=200, size=30):
    local = np.random.default_rng(seed=seed)
    draws = local.normal(loc=60, scale=8, size=(n, size)).mean(axis=1)
    return round(float(draws.mean()), 3), round(float(draws.std(ddof=1)), 3)


print(sample_means(1))
print(sample_means(1))
print(sample_means(2))

## 7. Exercises

**Exercise 1.** From `df`, print the two students with the highest score, showing only name and score.

**Exercise 2.** Compute the correlation between `hours` and `score` and say whether it is strong.

**Exercise 3.** Add a `passed` column (score ≥ 60) and print the pass rate.

**Exercise 4.** Draw a histogram of `hours` with 8 bins and a vertical line at the mean.

In [ ]:
# Exercise 1
print(df.nlargest(2, "score")[["student", "score"]].to_string(index=False))

In [ ]:
# Exercise 2
r = df["hours"].corr(df["score"])
print("r =", round(float(r), 3))

In [ ]:
# Exercise 3
df["passed"] = df["score"] >= 60
print("pass rate:", round(float(df["passed"].mean()), 3))

In [ ]:
# Exercise 4
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(df["hours"], bins=8, edgecolor="white")
ax.axvline(df["hours"].mean(), color="crimson", linestyle="--", label="mean")
ax.set_xlabel("Hours per week")
ax.set_ylabel("Students")
ax.set_title("Study hours with the mean marked")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Check yourself

1. What is the shape of `X.mean(axis=0)` for a `(6, 3)` matrix, and what does it mean?
2. Why does `[1, 2, 3] * 2` differ from `np.array([1, 2, 3]) * 2`?
3. Which of `df["score"] > 80` and `df[df["score"] > 80]` gives counts, and which gives rows?

Next: [LP02-01](../../P02-mathematics-for-ai/lessons/LP02-01-scalars-vectors-matrices-and-tensors.md).